# Module 3: Sequential Chain

Apply **Pattern 1**: break the single-agent ceiling by building a 3-stage pipeline where each agent has one focused job and passes its output to the next.

![Sequential Chain: Decision Brief → Researcher (gathers data with tools) → Analyst (evaluates options A/B/C) → Synthesizer (writes executive memo)](./architecture.png)

**When to use this pattern:**
- Steps have a natural, fixed order
- Each stage depends on the previous
- You want simple, predictable, debuggable flow

**Prerequisites:** Complete Module 1 first: this module reuses its tools.

## Tools Used in This Module

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `get_company_data(company_name)` | Returns NovaCart financial/operational data | CLV, churn rate, revenue, top-spender metrics |
| `get_market_benchmarks(industry)` | Returns e-commerce benchmarks | Avg CLV, churn, subscription rates, CLV lift range |
| `get_competitor_data(competitor_name)` | Returns competitor premium tier data | Pricing, pilot approach, adoption %, CLV lift, payback period |

> **Note:** Only the **Researcher** agent has access to these tools. The Analyst and Synthesizer work with the text handed to them by the previous stage: they have no tool access.

In [ ]:
%pip install -r requirements.txt

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1, Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2, Claude Haiku 4.5 (faster):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3, Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# Option 4, Amazon Nova Lite (cheapest):
#   model = BedrockModel(model_id="amazon.nova-lite-v1:0")
#
# Pass model= to Agent(...) to activate. Without it, Strands uses Claude Sonnet 4.

---

## Part 1: Import Tools and Define Prompts

We reuse the three mock tools from Module 1. Each agent gets a **narrow system prompt** focused on exactly one role. The key insight: the docstring tells the model *when* to use a tool; the system prompt tells it *what job it has*.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "02-single-agent"))

from strands import Agent
from strands.multiagent import GraphBuilder
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data

print("Tools loaded: get_company_data | get_market_benchmarks | get_competitor_data")

In [ ]:
# ── System prompts, each agent has ONE job ───────────────────────────────
# Narrow, focused prompts are the key to quality in a sequential chain.
# Each agent reads only what it needs and returns only what the next needs.

RESEARCHER_PROMPT = '''You are a market research specialist.
Given a decision brief, gather relevant company data, market benchmarks,
and competitor intelligence using your tools.
Return structured findings. data only, no recommendations.'''

ANALYST_PROMPT = '''You are a business strategy analyst.
Given market research findings and a decision brief, analyze each option (A, B, C).
For each option return:
- Strengths and weaknesses
- Implementation complexity: Low / Medium / High (with one-sentence justification)
- Top 2 risks with specific mitigations
- Verdict: Proceed / Proceed with caution / Do not proceed
Return structured analysis only. no executive memo yet.'''

SYNTHESIZER_PROMPT = '''You are an executive communications specialist.
Given research findings, option analyses, and the original brief, write a leadership memo:

## Decision Memo: [Title]
**Recommendation**: [one sentence: which option and why]

### Options at a Glance
| | Option A | Option B | Option C |
|---|---|---|---|
| Complexity | | | |
| Risk level | | | |
| Verdict | | | |

### Top 3 Risks & Mitigations
### Success Metrics (3-5 KPIs with targets)
### Decision Required: owner · deadline · approvers needed

Under 400 words. Be direct.'''

---

## Part 2: Create the Three Agents

Each agent gets a **narrow system prompt** focused on exactly one role.

`callback_handler=None` makes every agent **silent** during graph execution.
The GraphBuilder runs all three internally; we display the final output after the chain completes.

In [ ]:
import time

researcher = Agent(
    tools=[get_company_data, get_market_benchmarks, get_competitor_data],
    system_prompt=RESEARCHER_PROMPT,
    callback_handler=None,
)

analyst = Agent(
    system_prompt=ANALYST_PROMPT,
    callback_handler=None,
)

synthesizer = Agent(
    system_prompt=SYNTHESIZER_PROMPT,
    callback_handler=None,
)

print("Agents created: researcher | analyst | synthesizer")

---

## Part 3: Build the Graph and Run the Chain

`GraphBuilder` is the Strands primitive for **deterministic sequential workflows** (Workflow / DAG).

Define nodes and edges — the graph engine handles execution order and passes each node's output as input to the next node automatically.

In [ ]:
DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Company: NovaCart (2M active users, mid-size e-commerce)
Decision owners: VP Product + CFO approval required

Options to evaluate:
  Option A: Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B: Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C: Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

# ── Build the sequential graph ─────────────────────────────────────────
# GraphBuilder defines the DAG: nodes are agents, edges define execution order.
# Each node's output is automatically passed as input to the next node.
builder = GraphBuilder()
builder.add_node(researcher,  "researcher")
builder.add_node(analyst,     "analyst")
builder.add_node(synthesizer, "synthesizer")
builder.add_edge("researcher", "analyst")      # researcher → analyst
builder.add_edge("analyst",    "synthesizer")  # analyst → synthesizer

# ── Run the chain ─────────────────────────────────────────────────────
print("Running: Researcher → Analyst → Synthesizer")
print("─" * 60)

t0 = time.time()
graph_result = builder.build()(DECISION_BRIEF)
elapsed = time.time() - t0

print(f"\nChain complete in {elapsed:.1f}s")
print("─" * 60)

# ── Display the final memo ────────────────────────────────────────────
for node in reversed(graph_result.execution_order):
    if node.node_id == "synthesizer":
        print(str(node.result))
        break

---

## Part 4: Inspect the Pipeline

In [ ]:
# ── Inspect what each node produced ──────────────────────────────────────
for node in graph_result.execution_order:
    preview = str(node.result)[:300]
    print(f"=== {node.node_id.upper()} (first 300 chars) ===")
    print(preview, "...")
    print()

print("=== EXECUTION ORDER ===")
print(" → ".join(n.node_id for n in graph_result.execution_order))
print()
print("Notice: each agent only sees what it needs.")
print("The Researcher only sees the brief + tool results.")
print("The Analyst only sees brief + research.")
print("The Synthesizer only sees brief + research + analysis.")
print("No agent is asked to do everything: that is the pattern.")

In [ ]:
# ── Why this beats one agent doing everything ─────────────────────────────
# In Module 2 the single-agent ceiling had:
#   - all 4 roles (researcher + 3 analysts + synthesizer) in ONE context
#   - context that grew unbounded with every tool call
#
# Here, via GraphBuilder:
#   - Each agent has ONE focused role
#   - A context that only grows with what it needs
#   - Deterministic, debuggable execution order defined by graph edges
#
# The GraphBuilder (Workflow/DAG) is the right Strands primitive for this pattern.
# It replaces manual Python orchestration with a structured graph definition.

print("=== GRAPH TOPOLOGY ===")
print("researcher --> analyst --> synthesizer")
print()
print("Nodes:", len(graph_result.execution_order))
print("Execution order:", [n.node_id for n in graph_result.execution_order])

In [ ]:
# ── Token usage per node ─────────────────────────────────────────────────────
print(f"{'Stage':<14} {'Output (chars)':>16}")
print("-" * 32)
for node in graph_result.execution_order:
    chars = len(str(node.result))
    print(f"{node.node_id:<14} {chars:>16}")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| Sequential Chain | `GraphBuilder` defines the DAG; edges enforce execution order |
| `add_node` / `add_edge` | Declarative graph definition — no manual Python orchestration |
| Output propagation | Each node's output is automatically passed as input to the next |
| Focused context | Each agent only sees what it needs — no context bloat |
| Predictable flow | Fixed order, deterministic and debuggable step by step |

**Strands primitive:** `GraphBuilder` (Workflow / DAG)

---

## What's Next

In **Module 4: Parallel Fork-Join**, the three analysts run at the same time instead of in sequence. Same graph primitive, parallel edges — latency drops sharply.

---

## Want a real multi-turn conversation?

```bash
cd samples/03-sequential-chain
pip install -r requirements.txt
python chat.py
```